# Driven, damped oscillator: resonance

Left: Taylor's Figure 5.16, the steady-state $A^2$ against the natural frequency $\omega_0$, with the drive frequency $\omega$ fixed (tuning a radio).
Right: Taylor's Figure 5.15, the drive $f(t) = f_0\cos\omega t$ and the motion $x(t)$ it produces, starting from rest at $x = 0$.

Units: the drive has period 1 ($\omega = 2\pi$), and $f_0 = \omega^2$, so an amplitude of 1 is how far a steady push of $f_0$ would stretch a spring with $\omega_0 = \omega$. The sliders are $\omega_0/\omega$ and $\beta/\omega$.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

omega = 2 * np.pi          # drive frequency: period 1
f0 = omega**2              # so amplitudes are in units of f0/omega^2
x0, v0 = 0.0, 0.0          # starts at rest at x = 0 (change these for a different messy start)
t = np.linspace(0, 8, 2000)
w0s = np.linspace(0.01, 3, 1000) * omega        # natural frequencies for the left plot

## The two figures

The steady state is $x_p = A\cos(\omega t - \delta)$ with
$A^2 = \dfrac{f_0^2}{(\omega_0^2 - \omega^2)^2 + (2\beta\omega)^2}$ and $\delta = \tan^{-1}\dfrac{2\beta\omega}{\omega_0^2 - \omega^2}$ (Taylor 5.64-5.65).
The transient $x_h$ is last class's damped solution, with constants chosen so that $x_h + x_p$ starts at $x_0$, $v_0$.

In [ ]:
def A_and_delta(w0, beta):
    A = f0 / np.sqrt((w0**2 - omega**2)**2 + (2 * beta * omega)**2)
    delta = np.arctan2(2 * beta * omega, w0**2 - omega**2)    # arctan2 keeps delta between 0 and pi
    return A, delta

def x_h(xh0, vh0, beta, w0, t):
    # last class's damped solution starting at xh0, vh0 (any damping)
    if np.isclose(beta, w0):                                  # critically damped
        return (xh0 + (vh0 + beta * xh0) * t) * np.exp(-beta * t)
    q = np.sqrt(complex(beta**2 - w0**2))
    C1 = (vh0 + (beta + q) * xh0) / (2 * q)
    C2 = -(vh0 + (beta - q) * xh0) / (2 * q)
    return (np.exp(-beta * t) * (C1 * np.exp(q * t) + C2 * np.exp(-q * t))).real

def plot_resonance(omega0_over_omega=1.5, beta_over_omega=0.1, fixed_axes=True):
    w0, beta = omega0_over_omega * omega, beta_over_omega * omega
    A, delta = A_and_delta(w0, beta)
    xp = A * np.cos(omega * t - delta)
    # the transient makes up the difference between where x_p starts and where the cart starts
    x = xp + x_h(x0 - A * np.cos(delta), v0 - A * omega * np.sin(delta), beta, w0, t)

    fig = plt.figure(figsize=(13, 5))
    left = fig.add_subplot(1, 2, 1)
    drive = fig.add_subplot(4, 2, 2)
    motion = fig.add_subplot(4, 2, (4, 8))

    # Figure 5.16: A^2 against omega_0 at fixed omega
    left.plot(w0s / omega, A_and_delta(w0s, beta)[0]**2, 'k')
    left.plot([omega0_over_omega], [A**2], 'o', color='purple', markersize=10)
    left.axvline(1, color='gray', lw=0.5)
    left.set_xlabel(r'$\omega_0/\omega$'); left.set_ylabel(r'$A^2$')
    left.set_title(rf'$A^2 = {A**2:.2f}$ at $\omega_0 = {omega0_over_omega:.2f}\,\omega$; peak $= {(f0 / (2 * beta * omega))**2:.1f}$')

    # Figure 5.15: the drive and the motion
    drive.plot(t, np.cos(omega * t), "k")
    drive.set_ylabel(r'$f(t)/f_0$'); drive.set_xticklabels([])
    motion.plot(t, x, 'purple', label='x(t)')
    motion.plot(t, xp, color='gray', ls='--', lw=1, label='steady state')
    motion.axhline(0, color='gray', lw=0.5)
    motion.set_xlabel('t (drive periods)'); motion.set_ylabel('x')
    motion.legend(loc='upper right', frameon=False)

    if fixed_axes:   # fixed scales, so growing and shrinking show up as growing and shrinking
        left.set_ylim(0, 40)
        motion.set_ylim(-6.5, 6.5)
    plt.show()

interact(plot_resonance,
         omega0_over_omega=FloatSlider(value=1.5, min=0.2, max=3, step=0.05, description='ω0/ω'),
         beta_over_omega=FloatSlider(value=0.1, min=0.08, max=0.5, step=0.01, description='β/ω'),
         fixed_axes=True);

## Things to try

- Start at $\omega_0 = 1.5\,\omega$ and slide toward 1. Then past it.
- At $\omega_0 = \omega$, change $\beta$ from 0.1 to 0.2. The peak height should drop by a factor of 4 (it goes as $1/\beta^2$).
- Far from resonance ($\omega_0 = 3\omega$) the steady motion is tiny on the fixed scale: untick `fixed_axes` to zoom in.
- Watch the messy start: while $\beta < \omega_0$, how long it lasts depends on $\beta$ only, not on $\omega_0$ (it dies as $e^{-\beta t}$).
- The messiest start, most like Taylor's Figure 5.15: $\omega_0 = 3\,\omega$ with `fixed_axes` unticked.

*(your notes here)*